# Application Scenario 1: NDVI Calculation

This scenario exports the output to the `output` directory.\
Please clear this directory between pipeline executions.

## Download the resources

- configuration file

In [ ]:
!curl -sL https://raw.githubusercontent.com/geospaitial-lab/aviary-ACM-SIGSPATIAL-2026/main/application_scenarios/1/config.yaml -o config.yaml
!mkdir output

## Installation

### With `pip`

In [ ]:
!pip install geospaitial-lab-aviary[cli,expression] ipywidgets

### With `uv`

In [ ]:
!uv pip install geospaitial-lab-aviary[cli,expression] ipywidgets

---

## Python API

### Imports

In [ ]:
from pathlib import Path
import warnings

import aviary
import aviary.pipeline
import aviary.tile
import aviary.utils

### Check the version

In [ ]:
print(aviary.__version__)

### Suppress experimental warnings

Some components used in this scenario are experimental.

In [ ]:
warnings.filterwarnings('ignore', category=aviary.AviaryExperimentalWarning)

### Define the `Grid`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/core/grid)

In [ ]:
bounding_box = aviary.BoundingBox(
    x_min=362000,
    y_min=5714500,
    x_max=364000,
    y_max=5716500,
)

grid = aviary.Grid.from_bounding_box(
    bounding_box=bounding_box,
    tile_size=500,
)

### Define the `TileFetcher`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/tile/tile_fetcher/tile_fetcher)

In [ ]:
wms_fetcher_rgb = aviary.tile.WMSFetcher(
    url='https://www.wms.nrw.de/geobasis/wms_nw_dop',
    version=aviary.WMSVersion.V1_3_0,
    layer='nw_dop_rgb',
    epsg_code=25832,
    response_format='image/png',
    channel_names=['r', None, None],
    tile_size=500,
    ground_sampling_distance=.2,
)

wms_fetcher_nir = aviary.tile.WMSFetcher(
    url='https://www.wms.nrw.de/geobasis/wms_nw_dop',
    version=aviary.WMSVersion.V1_3_0,
    layer='nw_dop_nir',
    epsg_code=25832,
    response_format='image/png',
    channel_names=['nir', None, None],
    tile_size=500,
    ground_sampling_distance=.2,
)

composite_fetcher = aviary.tile.CompositeFetcher(
    tile_fetchers=[wms_fetcher_rgb, wms_fetcher_nir],
)

### Define the `TilesProcessor`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/tile/tiles_processor/tiles_processor)

In [ ]:
ndvi_processor = aviary.tile.ExpressionProcessor(
    expression_string='(nir - r) / (nir + r)',
    new_channel_name='ndvi',
    dtype=aviary.DType.FLOAT32,
)

raster_exporter = aviary.tile.RasterExporter(
    channel_names='ndvi',
    epsg_code=25832,
    path=Path('output/ndvi'),
)

composite_processor = aviary.tile.SequentialCompositeProcessor(
    tiles_processors=[ndvi_processor, raster_exporter],
)

### Define the `Logger`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/utils/logger)

In [ ]:
logger = aviary.utils.Logger(
    sink='output/application_scenario_1.log',
    level=aviary.LogLevel.TRACE,
)

### Define the `TilePipeline`

[Docs](https://geospaitial-lab.github.io/aviary/api_reference/pipeline/pipeline)

In [ ]:
tile_pipeline = aviary.pipeline.TilePipeline(
    grid=grid,
    tile_fetcher=composite_fetcher,
    tiles_processor=composite_processor,
    tile_loader_batch_size=4,
    tile_loader_max_num_threads=None,
    tile_loader_num_prefetched_tiles=1,
    show_progress=True,
)

### Execute the `TilePipeline`

In [ ]:
tile_pipeline()

---

## CLI

### Check the version

In [ ]:
!aviary --version

### Suppress experimental warnings

Some components used in this scenario are experimental.

In [ ]:
%env AVIARY_EXPERIMENTAL_WARNINGS=false

### Validate the configuration file

[Docs](https://geospaitial-lab.github.io/aviary/cli_reference/aviary_pipeline/pipeline_validate)

In [ ]:
!aviary pipeline validate config.yaml

### Set the path to the log file and the log level

In [ ]:
%env AVIARY_LOG_PATH=output/application_scenario_1.log
%env AVIARY_LOG_LEVEL=trace

### Execute the `TilePipeline`

[Docs](https://geospaitial-lab.github.io/aviary/cli_reference/aviary_pipeline/pipeline_run)

In [ ]:
!aviary pipeline run config.yaml